In [ ]:
# Setup: garante que .env seja carregado e pacote esteja no path
import sys, os
from pathlib import Path

# Localiza raiz do projeto (sobe até encontrar pyproject.toml)
project_root = Path(os.getcwd())
for p in [project_root, *project_root.parents]:
    if (p / 'pyproject.toml').exists():
        project_root = p
        break

# Adiciona src/ ao sys.path (caso o pacote não esteja instalado em editable mode)
src_path = str(project_root / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Carrega .env da raiz do projeto
from dotenv import load_dotenv
load_dotenv(project_root / '.env', override=False)

print(f'Project root: {project_root}')
print(f'FRED_API_KEY: {"SET" if os.getenv("FRED_API_KEY") else "NOT SET — verifique o .env"}')

# Analise de Acoes e Fundamentos

Analise de acoes brasileiras: precos, dados fundamentalistas, dividendos e setores.

Fontes:
- **YahooFinanceFetcher**: precos OHLCV, fundamentos (P/L, P/VP, ROE, DY), dividendos
- **CVMFetcher**: demonstracoes financeiras oficiais (DFP, ITR)
- **DDMFetcher**: cotacoes, indicadores e FIIs (requer API key)
- **MarketSectorAnalyzer**: performance setorial
- **EconomicSectorAnalyzer**: setores da economia real (IBGE)

**Nota**: Yahoo Finance nao requer API key. CVM e livre mas endpoints podem estar indisponiveis.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

# Tickers de exemplo
TICKERS = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "WEGE3.SA"]

## 1. Precos Historicos — Yahoo Finance

In [ ]:
from carteira_auto.data.fetchers import YahooFinanceFetcher

yahoo = YahooFinanceFetcher()

# Precos atuais
precos = yahoo.get_multiple_prices(TICKERS)
print("Precos atuais:")
for ticker, preco in precos.items():
    print(f"  {ticker}: R${preco:.2f}" if preco else f"  {ticker}: indisponivel")

In [ ]:
# Historico 1 ano
historico = yahoo.get_historical_price_data(TICKERS, period="1y")
print(f"Shape: {historico.shape}")

# Plotar retorno acumulado normalizado
fig, ax = plt.subplots()
for ticker in TICKERS:
    try:
        if isinstance(historico.columns, pd.MultiIndex):
            close = historico[(ticker, "Close")].dropna()
        else:
            close = historico["Close"].dropna()
        retorno_norm = close / close.iloc[0] * 100
        ax.plot(retorno_norm.index, retorno_norm.values, label=ticker.replace(".SA", ""))
    except (KeyError, TypeError):
        pass

ax.axhline(y=100, color="gray", linestyle="--", alpha=0.5)
ax.set_title("Retorno Normalizado (base 100) — Ultimo Ano")
ax.set_ylabel("Base 100")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Dados Fundamentalistas

In [ ]:
# Multiplos fundamentalistas via get_batch_info
campos = [
    "trailingPE", "priceToBook", "dividendYield",
    "returnOnEquity", "debtToEquity", "marketCap"
]
info = yahoo.get_batch_info(TICKERS, fields=campos)

print("Multiplos Fundamentalistas:")
print(info.to_string())

In [ ]:
# Visualizar múltiplos fundamentalistas
if not info.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    tickers_label = [t.replace(".SA", "") for t in info.index]
    
    # P/L
    if "trailingPE" in info.columns:
        vals = info["trailingPE"].fillna(0)
        cores = ["#2ca02c" if v < 15 else "#ff7f0e" if v < 25 else "#d62728" for v in vals]
        axes[0].bar(tickers_label, vals, color=cores, alpha=0.85)
        axes[0].set_title("P/L (Trailing)")
        axes[0].axhline(y=15, color="gray", linestyle="--", alpha=0.4, label="P/L 15")
        axes[0].legend()
    
    # Dividend Yield
    if "dividendYield" in info.columns:
        vals = (info["dividendYield"].fillna(0) * 100)
        cores = ["#2ca02c" if v > 5 else "#ff7f0e" if v > 2 else "#d62728" for v in vals]
        axes[1].bar(tickers_label, vals, color=cores, alpha=0.85)
        axes[1].set_title("Dividend Yield (%)")
    
    # ROE
    if "returnOnEquity" in info.columns:
        vals = (info["returnOnEquity"].fillna(0) * 100)
        cores = ["#2ca02c" if v > 15 else "#ff7f0e" if v > 5 else "#d62728" for v in vals]
        axes[2].bar(tickers_label, vals, color=cores, alpha=0.85)
        axes[2].set_title("ROE (%)")
    
    plt.suptitle("Múltiplos Fundamentalistas", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
# Info detalhada de um ativo
petr4_info = yahoo.get_basic_info("PETR4.SA")
print("PETR4 — Informacoes basicas:")
for key in ["shortName", "sector", "industry", "marketCap", "trailingPE", "dividendYield"]:
    val = petr4_info.get(key, "N/A")
    print(f"  {key}: {val}")

## 3. Dividendos

In [ ]:
# Histórico de dividendos
divs = yahoo.get_dividends("PETR4.SA")
print(f"PETR4 dividendos: {len(divs)} pagamentos")
if not divs.empty:
    print(divs.tail(10))

    # Total últimos 12 meses
    from datetime import datetime, timedelta
    cutoff = datetime.now() - timedelta(days=365)
    divs_12m = divs[divs.index >= cutoff]
    print(f"\nTotal dividendos 12m: R${divs_12m.sum().iloc[0]:.2f}" if not divs_12m.empty else "")

In [ ]:
# Visualizar dividendos PETR4 (últimos pagamentos)
if not divs.empty:
    divs_recent = divs.tail(20)
    fig, ax = plt.subplots()
    ax.bar(range(len(divs_recent)), divs_recent.iloc[:, 0],
           color="#2ca02c", alpha=0.85)
    ax.set_xticks(range(len(divs_recent)))
    ax.set_xticklabels(
        [str(d.date()) for d in divs_recent.index],
        rotation=45, ha="right", fontsize=8
    )
    ax.set_title("PETR4 — Histórico de Dividendos (Últimos Pagamentos)")
    ax.set_ylabel("R$/ação")
    plt.tight_layout()
    plt.show()

## 4. CVM — Demonstracoes Financeiras Oficiais

In [ ]:
from carteira_auto.data.fetchers import CVMFetcher

cvm = CVMFetcher()

# Cadastro de empresas
cadastro = cvm.get_company_registry()
print(f"Empresas na CVM: {len(cadastro)}")

# Mapeamento ticker -> CNPJ (util para buscar DFP por ticker)
try:
    mapa = cvm.build_ticker_cnpj_map()
    print(f"Mapa ticker->CNPJ: {len(mapa)} tickers")
    for t in ["PETR4", "VALE3", "ITUB4"]:
        cnpj = mapa.get(t, "nao encontrado")
        print(f"  {t}: {cnpj}")
except Exception as e:
    print(f"Erro ao construir mapa: {e}")

In [ ]:
# DFP — Demonstracao do Resultado (DRE)
try:
    dre = cvm.get_dfp_by_ticker("PETR4", year=2023, statement="DRE")
    print(f"DRE Petrobras 2023: {len(dre)} linhas")
    print(dre.head(10))
except Exception as e:
    print(f"CVM DFP indisponivel: {e}")
    print("Nota: endpoint CVM pode estar temporariamente fora do ar")

## 5. DDM — Dados de Mercado (API paga)

In [ ]:
DDM_OK = bool(os.getenv("DADOS_MERCADO_API_KEY"))
print(f"DDM API key configurada: {DDM_OK}")

if DDM_OK:
    from carteira_auto.data.fetchers import DDMFetcher
    ddm = DDMFetcher()

    # Cotacoes
    cotacoes = ddm.get_quotations("PETR4", period_init="2024-01-01")
    print(f"Cotacoes PETR4: {len(cotacoes)} dias")

    # Indicadores de mercado
    indices = ddm.get_market_indices()
    print(f"Indices: {len(indices)} registros")

    # Lista de FIIs
    fiis = ddm.get_fii_list()
    print(f"FIIs listados: {len(fiis)}")

## 6. Analise Setorial

In [ ]:
from carteira_auto.analyzers import MarketSectorAnalyzer, EconomicSectorAnalyzer
from carteira_auto.core.engine import PipelineContext

# Setores de mercado (Yahoo Finance)
msa = MarketSectorAnalyzer()
ctx = PipelineContext()
ctx = msa.run(ctx)

market_sectors = ctx.get("market_sectors", [])
if market_sectors:
    print(f"Setores de mercado: {len(market_sectors)} itens")
    for s in market_sectors[:10]:
        ret = f"{s.return_pct:+.2f}%" if s.return_pct is not None else "N/A"
        print(f"  {s.sector:<30s} {ret}")
else:
    print("Sem dados de setores de mercado")

if ctx.has_errors:
    print(f"\nErros parciais: {ctx.errors}")

In [ ]:
# Setores da economia real (IBGE) — EconomicSectorAnalyzer
esa = EconomicSectorAnalyzer()
ctx2 = PipelineContext()
ctx2 = esa.run(ctx2)

eco_sectors = ctx2.get("economic_sectors", [])
if eco_sectors:
    print(f"Setores econômicos IBGE: {len(eco_sectors)} registros")
    for s in eco_sectors:
        gr = f"{s.growth_rate:+.1f}%" if s.growth_rate is not None else "N/A"
        print(f"  {gr:8s}  {s.sector[:60]}")
else:
    print("Sem dados de setores econômicos")

if ctx2.has_errors:
    print(f"\nErros parciais: {ctx2.errors}")

In [ ]:
# Gráfico: crescimento por setor econômico (IBGE)
if eco_sectors:
    setores_eco = [s for s in eco_sectors if s.growth_rate is not None]
    if setores_eco:
        labels_eco = [s.sector[:40] for s in setores_eco]
        vals_eco = [s.growth_rate for s in setores_eco]
        cores_eco = ["#2ca02c" if v >= 0 else "#d62728" for v in vals_eco]

        fig, ax = plt.subplots(figsize=(10, max(4, len(setores_eco) * 0.5)))
        bars = ax.barh(labels_eco, vals_eco, color=cores_eco, alpha=0.85)
        ax.axvline(x=0, color="black", linewidth=0.8)
        for bar, val in zip(bars, vals_eco, strict=False):
            offset = 0.05 if val >= 0 else -0.15
            ax.text(val + offset, bar.get_y() + bar.get_height() / 2,
                    f"{val:+.1f}%", va="center", fontsize=9)
        ax.set_title("Crescimento por Setor Econômico — EconomicSectorAnalyzer (IBGE)")
        ax.set_xlabel("% crescimento trimestral")
        plt.tight_layout()
        plt.show()
else:
    print("Sem dados de setores econômicos para gráfico")

In [ ]:
# Setores da economia real (IBGE)
esa = EconomicSectorAnalyzer()
ctx2 = PipelineContext()
ctx2 = esa.run(ctx2)

if "economic_sectors" in ctx2:
    print("Setores economicos (IBGE):")
    for sector in ctx2["economic_sectors"]:
        print(f"  {sector.name}: {sector.value}")

if ctx2.has_errors:
    print(f"\nErros parciais: {ctx2.errors}")

## Resumo

| Fonte | Dados | Uso |
|-------|-------|-----|
| Yahoo Finance | Precos, fundamentos, dividendos | Analise quantitativa diaria |
| CVM | DFP, DRE oficiais | Validacao de fundamentos |
| DDM | Cotacoes, indicadores, FIIs | Complemento com dados nacionais |
| MarketSectorAnalyzer | Performance setorial | Alocacao setorial |
| EconomicSectorAnalyzer | Crescimento real (IBGE) | Contexto macrosetorial |